# Hybrid model selection

Fit every model from `final_models`, append its forecast to the data, and train a LightGBM router to select the base model observation by observation. The router is trained on chronological out-of-fold forecasts so that its labels do not benefit from in-sample base-model fit.

In [9]:
import numpy as np
import pandas as pd
from IPython.display import display

from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import ElasticNet, HuberRegressor, Lasso, LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

## Data preparation

This is the same preparation used in `final_models`. Train and test are combined only while creating backward-looking features, then separated again.

In [10]:
train_data = pd.read_parquet("../data/train.parquet").sort_values("date").reset_index(drop=True)
test_data = pd.read_parquet("../data/test.parquet").sort_values("date").reset_index(drop=True)

for data in (train_data, test_data):
    data["usd_zar_28_movement"] = data["usd_zar_28"] - data["usd_zar"]

all_data = (
    pd.concat(
        [train_data.assign(_split="train"), test_data.assign(_split="test")],
        ignore_index=True,
    )
    .sort_values("date")
    .reset_index(drop=True)
)
all_data["interest_rate_diff"] = all_data["sa_repo_rate"] - all_data["us_fed_funds"]
all_data["policy_rate_differential"] = all_data["interest_rate_diff"]
all_data["sa_us_5y_yield_spread"] = all_data["sa_5y_yield"] - all_data["us_5y_yield"]

log_usd_zar = np.log(all_data["usd_zar"])
all_data["fx_log_return_5d"] = log_usd_zar.diff(5)
all_data["fx_log_return_21d"] = log_usd_zar.diff(21)
all_data["fx_realized_vol_21d"] = log_usd_zar.diff().shift(1).rolling(21).std()
all_data["fx_realized_vol_63d"] = log_usd_zar.diff().shift(1).rolling(63).std()
all_data["sa_cpi_log_change_21d"] = np.log(all_data["sa_cpi"]).diff(21)
all_data["sa_real_gdp_log_change_63d"] = np.log(all_data["sa_real_gdp"]).diff(63)
all_data["commodities"] = all_data[
    ["iron_ore_usd_per_tonne", "gold_usd_per_oz", "platinum_usd_per_oz", "richards_bay_coal_usd"]
].mean(axis=1)

engineered_train = all_data.loc[all_data["_split"].eq("train")].drop(columns="_split").reset_index(drop=True)
engineered_test = all_data.loc[all_data["_split"].eq("test")].drop(columns="_split").reset_index(drop=True)
y_train = engineered_train["usd_zar_28_movement"]
y_test = engineered_test["usd_zar_28_movement"]

## Base-model specifications

Each factory below reproduces the features and hyperparameters in `final_models`. Factories are used so the same specification can be fitted independently in every time-series fold and once more on the full training set.

In [11]:
MODEL_SPECS = {
    "LightGBM": (
        ["gold_usd_per_oz", "usd_zar_1w_return", "usd_zar_1m_volatility", "interest_rate_diff"],
        lambda: LGBMRegressor(n_estimators=150, learning_rate=0.05, num_leaves=15, random_state=42, verbosity=-1, n_jobs=1),
    ),
    "XGBoost": (
        ["gold_usd_per_oz", "usd_zar_1w_return", "usd_zar_1m_volatility", "interest_rate_diff"],
        lambda: XGBRegressor(objective="reg:squarederror", n_estimators=500, learning_rate=0.03, max_depth=3, min_child_weight=5, gamma=0.1, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0, random_state=42, n_jobs=-1),
    ),
    "Lasso": (
        ["usd_zar", "brent_usd_per_barrel", "interest_rate_diff", "sa_us_5y_yield_spread", "sa_yoy_inflation", "sa_5y_cds_bp", "vix", "broad_usd_index", "sa_cpi", "usd_zar_1w_return", "usd_zar_1m_return", "usd_zar_3m_return", "usd_zar_1m_volatility"],
        lambda: make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=50_000)),
    ),
    "MLP": (
        ["gold_usd_per_oz_return", "sa_yoy_inflation", "usd_zar_1m_return", "usd_zar_1m_volatility"],
        lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(128, 64, 32), alpha=0.01, learning_rate="adaptive", max_iter=1000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=30, random_state=42)),
    ),
    "OLS": (
        ["broad_usd_index", "brent_usd_per_barrel", "sa_cpi", "usd_zar_1m_return", "fx_log_return_5d", "fx_log_return_21d", "sa_cpi_log_change_21d", "sa_real_gdp_log_change_63d"],
        lambda: make_pipeline(StandardScaler(), LinearRegression()),
    ),
    "Ridge": (
        ["sa_repo_rate", "sa_real_gdp", "policy_rate_differential", "sa_us_5y_yield_spread", "sa_cpi_log_change_21d"],
        lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    ),
    "Elastic Net": (
        ["sa_real_gdp", "sa_5y_cds_bp", "sa_5y_yield", "policy_rate_differential", "fx_log_return_5d", "fx_realized_vol_21d"],
        lambda: TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), ElasticNet(alpha=0.02, l1_ratio=0.35, max_iter=20_000, random_state=42)), transformer=StandardScaler()),
    ),
    "Huber": (
        ["sa_repo_rate", "usd_zar_1m_return", "sa_cpi_log_change_21d"],
        lambda: TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), HuberRegressor(epsilon=1.35, alpha=0.01, max_iter=2_000)), transformer=StandardScaler()),
    ),
    "Spline Ridge": (
        ["iron_ore_usd_per_tonne", "sa_repo_rate", "sa_cpi", "sa_yoy_inflation", "usd_zar_1w_return", "usd_zar_1m_return", "sa_5y_yield", "interest_rate_diff", "fx_realized_vol_63d"],
        lambda: make_pipeline(StandardScaler(), SplineTransformer(n_knots=4, degree=2, include_bias=False), Ridge(alpha=25.0)),
    ),
}

MODEL_NAMES = list(MODEL_SPECS)
ROUTER_FEATURES = sorted({feature for features, _ in MODEL_SPECS.values() for feature in features})
PREDICTION_COLUMNS = {name: f"prediction_{name.lower().replace(' ', '_')}" for name in MODEL_NAMES}
print(f"Router uses the union of {len(ROUTER_FEATURES)} predictors across {len(MODEL_NAMES)} models.")

Router uses the union of 26 predictors across 9 models.


## Leakage-safe training data for the router

Within each chronological split, every base model is fitted only on earlier observations. Its forecast is then appended to the held-out rows. The router's class label is the model with the smallest absolute forecast error on that row. Because squared error has the same row-wise ordering, this label is aligned with the final RMSE objective.

In [12]:
def fit_base_models(data, target):
    fitted = {}
    for name, (features, factory) in MODEL_SPECS.items():
        fit_rows = data[features].notna().all(axis=1) & target.notna()
        fitted[name] = factory().fit(data.loc[fit_rows, features], target.loc[fit_rows])
    return fitted


def append_model_predictions(data, fitted_models):
    result = data.copy()
    for name, model in fitted_models.items():
        features = MODEL_SPECS[name][0]
        valid = result[features].notna().all(axis=1)
        result[PREDICTION_COLUMNS[name]] = np.nan
        result.loc[valid, PREDICTION_COLUMNS[name]] = model.predict(result.loc[valid, features])
    return result


oof_train = engineered_train.copy()
for column in PREDICTION_COLUMNS.values():
    oof_train[column] = np.nan

time_splits = TimeSeriesSplit(n_splits=5)
for fold, (fit_index, validation_index) in enumerate(time_splits.split(engineered_train), start=1):
    fold_models = fit_base_models(engineered_train.iloc[fit_index], y_train.iloc[fit_index])
    fold_predictions = append_model_predictions(engineered_train.iloc[validation_index], fold_models)
    oof_train.loc[validation_index, list(PREDICTION_COLUMNS.values())] = fold_predictions[list(PREDICTION_COLUMNS.values())].to_numpy()
    print(f"Finished fold {fold}")

prediction_columns = list(PREDICTION_COLUMNS.values())
router_rows = (
    oof_train[ROUTER_FEATURES + prediction_columns].notna().all(axis=1)
    & y_train.notna()
)
oof_errors = oof_train.loc[router_rows, prediction_columns].sub(y_train.loc[router_rows], axis=0).abs()
router_target = oof_errors.to_numpy().argmin(axis=1)

router_model = LGBMClassifier(
    objective="multiclass",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=15,
    random_state=42,
    verbosity=-1,
    n_jobs=1,
)
router_model.fit(oof_train.loc[router_rows, ROUTER_FEATURES], router_target)

pd.Series(np.asarray(MODEL_NAMES)[router_target], name="best_model").value_counts().rename("OOF wins").to_frame()

Finished fold 1
Finished fold 2
Finished fold 3
Finished fold 4
Finished fold 5


,OOF wins
best_model,
MLP,430
LightGBM,414
OLS,348
Ridge,300
XGBoost,260
Elastic Net,257
Huber,251
Lasso,198
Spline Ridge,197


## Fit the final base models and append their responses

These are the deployable base models: each is now fitted on the complete training set, exactly as in `final_models`. Their responses are retained in both dataframes for inspection.

In [13]:
final_base_models = fit_base_models(engineered_train, y_train)
hybrid_train_data = append_model_predictions(engineered_train, final_base_models)
hybrid_test_data = append_model_predictions(engineered_test, final_base_models)

hybrid_train_data[["date", *prediction_columns]].head()

,date,prediction_lightgbm,prediction_xgboost,prediction_lasso,prediction_mlp,prediction_ols,prediction_ridge,prediction_elastic_net,prediction_huber,prediction_spline_ridge
0,2008-10-10,0.334785,0.470954,0.017598,0.063101,NaN,NaN,NaN,NaN,NaN
1,2008-10-13,0.208689,0.470954,0.017598,0.063101,NaN,NaN,NaN,NaN,NaN
2,2008-10-14,0.005465,0.230645,0.028702,0.178213,NaN,NaN,NaN,NaN,NaN
3,2008-10-15,0.005465,0.230645,0.022230,0.161187,NaN,NaN,NaN,NaN,NaN
4,2008-10-16,-0.023959,-0.071365,0.019521,-0.417408,NaN,NaN,NaN,NaN,NaN


## Hybrid prediction pipeline

The pipeline first obtains every base forecast, asks the LightGBM router which model applies to each observation, and returns that model's forecast. `routing_decisions` exposes the choices for analysis.

In [14]:
class HybridModelPipeline(BaseEstimator, RegressorMixin):
    def __init__(self, base_models, model_specs, router, router_features):
        self.base_models = base_models
        self.model_specs = model_specs
        self.router = router
        self.router_features = router_features
        self.model_names = list(model_specs)

    def base_predictions(self, X):
        return np.column_stack([
            self.base_models[name].predict(X[self.model_specs[name][0]])
            for name in self.model_names
        ])

    def routing_decisions(self, X):
        class_ids = self.router.predict(X[self.router_features]).astype(int)
        return pd.Series(np.asarray(self.model_names)[class_ids], index=X.index, name="selected_model")

    def predict(self, X):
        predictions = self.base_predictions(X)
        class_ids = self.router.predict(X[self.router_features]).astype(int)
        return predictions[np.arange(len(X)), class_ids]


hybrid_model = HybridModelPipeline(final_base_models, MODEL_SPECS, router_model, ROUTER_FEATURES)

for data in (hybrid_train_data, hybrid_test_data):
    valid = data[ROUTER_FEATURES].notna().all(axis=1)
    data["hybrid_selected_model"] = pd.NA
    data["hybrid_prediction"] = np.nan
    data.loc[valid, "hybrid_selected_model"] = hybrid_model.routing_decisions(data.loc[valid]).to_numpy()
    data.loc[valid, "hybrid_prediction"] = hybrid_model.predict(data.loc[valid])

hybrid_test_data["hybrid_selected_model"].value_counts().rename("test selections").to_frame()

,test selections
hybrid_selected_model,
MLP,658
Huber,195
OLS,170
LightGBM,109
Spline Ridge,97
XGBoost,38
Ridge,29
Elastic Net,5
Lasso,3


## Evaluation

As in `final_models`, compare movement RMSE against persistence and the training-fitted AR(1) benchmark on both train and test. The training score is descriptive and optimistic because the final base models saw those observations; the untouched test result is the primary hybrid-model estimate.

In [15]:
AR1_HORIZON = 28
train_log_usd_zar = np.log(engineered_train["usd_zar"])
ar1_model = LinearRegression().fit(
    train_log_usd_zar.shift(1).iloc[1:].to_frame("lagged_log_usd_zar"),
    train_log_usd_zar.iloc[1:],
)


def ar1_movement_forecast(spot, horizon=AR1_HORIZON):
    phi = float(ar1_model.coef_[0])
    intercept = float(ar1_model.intercept_)
    log_spot = np.log(np.asarray(spot))
    if np.isclose(phi, 1.0):
        log_forecast = log_spot + intercept * horizon
    else:
        log_forecast = phi**horizon * log_spot + intercept * (1 - phi**horizon) / (1 - phi)
    return np.exp(log_forecast) - np.asarray(spot)


def evaluate_model(model_name, model, features, data, target):
    valid_rows = data[features].notna().all(axis=1) & target.notna()
    actual = target.loc[valid_rows]
    model_prediction = model.predict(data.loc[valid_rows, features])
    persistence_prediction = np.zeros(len(actual))
    ar1_prediction = ar1_movement_forecast(data.loc[valid_rows, "usd_zar"])
    return pd.DataFrame(
        {"RMSE": [
            root_mean_squared_error(actual, model_prediction),
            root_mean_squared_error(actual, persistence_prediction),
            root_mean_squared_error(actual, ar1_prediction),
        ]},
        index=[model_name, "Persistence", "AR(1)"],
    ).sort_values("RMSE")


hybrid_train_results = evaluate_model("Hybrid", hybrid_model, ROUTER_FEATURES, engineered_train, y_train)
hybrid_test_results = evaluate_model("Hybrid", hybrid_model, ROUTER_FEATURES, engineered_test, y_test)

display(hybrid_train_results.style.set_caption("Training data"))
display(hybrid_test_results.style.set_caption("Test data"))

,RMSE
Hybrid,0.375618
AR(1),0.520351
Persistence,0.523211


,RMSE
AR(1),0.544117
Persistence,0.544335
Hybrid,0.598196
